<a href="https://colab.research.google.com/github/Veerababu77/House_Prediction_Regression/blob/main/House_Prediction_Regression_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
dataset = pd.read_csv('train.csv')
x = pd.read_csv('train.csv')
y = dataset.iloc[:, -1].values
dataset = dataset.drop(columns = ['SalePrice'])

In [ ]:
def dropping_columns(df, threshold):
  missing_pct = df.isnull().mean()
  return df.drop(columns = missing_pct[missing_pct > threshold].index)
dataset = dropping_columns(dataset, threshold = 0.50)

In [ ]:
dataset['has_garage'] = dataset['GarageYrBlt'].notnull().astype(int)
dataset['GarageYrBlt'] = dataset['GarageYrBlt'].fillna(x['YearBuilt'])

In [ ]:
print(dataset.shape)

(1460, 76)


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

categorical_missing_columns = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish','GarageQual', 'GarageCond']
numeric_missing_columns = ['LotFrontage','MasVnrArea']

dataset[numeric_missing_columns] = dataset[numeric_missing_columns].fillna(dataset[numeric_missing_columns].median())

for col in categorical_missing_columns:
    dataset[col] = dataset[col].fillna(dataset[col].mode()[0])

scaler = dataset
all_numeric_columns = dataset.select_dtypes(include = ['float64', 'int64']).columns



In [ ]:
dataset.dtypes, dataset.shape

(Id                 int64
 MSSubClass         int64
 MSZoning          object
 LotFrontage      float64
 LotArea            int64
                   ...   
 MoSold             int64
 YrSold             int64
 SaleType          object
 SaleCondition     object
 has_garage         int64
 Length: 76, dtype: object,
 (1460, 76))

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_columns = dataset.select_dtypes(include=['object', 'category']).columns
ct = ColumnTransformer(transformers=[
    ('encoder', OneHotEncoder(sparse_output=False), categorical_columns)
], remainder='passthrough')

dataset = ct.fit_transform(dataset)

vector = dataset

In [ ]:
dataset.dtype, dataset.shape

(dtype('float64'), (1460, 273))

In [ ]:
from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(dataset, y, test_size = 0.2, random_state = 0)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state = 0)
tree.fit(xtrain, ytrain)

DecisionTreeRegressor(random_state=0)

In [ ]:
tree_predict = tree.predict(xtest)

from sklearn.metrics import r2_score
r2_score(ytest, tree_predict)

0.7440365411678358

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(n_estimators = 20, random_state = 0)
forest.fit(xtrain, ytrain)

forest_predict = forest.predict(xtest)

r2_score(ytest, forest_predict)

0.8315193599513329

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()
regressor.fit(xtrain, ytrain)

linear_predict = regressor.predict(xtest)
r2_score(ytest, linear_predict)

0.5600676543646022

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state = 0
)
gbr = gbr.fit(xtrain, ytrain)

gbr_predict = gbr.predict(xtest)
r2_score(ytest, gbr_predict)

0.8931075180802787

In [ ]:

from xgboost import XGBRegressor

xbr = XGBRegressor(
    n_estimators = 200,
    learning_rate = 0.1,
    max_depth = 3,
    random_state = 42
)

xbr.fit(xtrain, ytrain)
xbr_predict = xbr.predict(xtest)
r2_score(ytest, xbr_predict)

0.8787442445755005

In [ ]:
from lightgbm import LGBMRegressor

lgb = LGBMRegressor(
    n_estimators = 100,
    learning_rate = 0.1,
    max_depth = -1,
    num_leaves = 31,
    random_state = 0
)

lgb.fit(xtrain, ytrain)
lgb_predict = lgb.predict(xtest)
r2_score(ytest, lgb_predict)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001721 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3421
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 171
[LightGBM] [Info] Start training from score 180808.898973


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


0.8420422640515093

In [ ]:
scaler.head()

!pip install catboost
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
sxtrain, sxtest, sytrain, sytest = train_test_split(scaler, y, test_size = 0.2, random_state = 0)

from sklearn.linear_model import Ridge, Lasso
categorical_columns = sxtrain.select_dtypes(include=['object']).columns.tolist()


cat_boost = CatBoostRegressor(
    iterations = 2000,
    learning_rate = 0.03,
    depth = 6,
    random_state = 42,
    verbose = 0
)

cat_boost.fit(
    sxtrain, sytrain,
    cat_features=categorical_columns,
    eval_set=(sxtest, sytest),
    early_stopping_rounds=50
)

print("Best iteration:", cat_boost.get_best_iteration())



Best iteration: 1597


In [ ]:
cat_predict = cat_boost.predict(sxtest)

r2_score(sytest, cat_predict)

0.8927154718082201